# LTX Video Test: Text-to-Video + Image-to-Video

Prueba rápida para verificar que [Lightricks/LTX-Video](https://huggingface.co/Lightricks/LTX-Video) funciona correctamente.

**Pipeline:** `LTXPipeline` (text-to-video) + `LTXConditionPipeline` (image-to-video)

**Requisitos:** T4 GPU (~10 GB VRAM), última versión de diffusers

In [ ]:
# ---- 1. INSTALAR DEPENDENCIAS ----
import os, torch
# LTX-Video requiere diffusers >= 0.30.0
!pip install -qU diffusers transformers accelerate sentencepiece
!pip install -q imageio[ffmpeg] pillow

In [ ]:
# ---- 2. IMPORTS ----
import torch
from diffusers import LTXPipeline, LTXConditionPipeline
from diffusers.utils import load_image, export_to_video
from diffusers.pipelines.ltx.pipeline_ltx_condition import LTXVideoCondition
from IPython.display import Video
import os, gc

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.bfloat16
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

## Text-to-Video con LTXPipeline

In [ ]:
# ---- 3. TEXT-TO-VIDEO ----

gc.collect()
torch.cuda.empty_cache()

print('Cargando LTXPipeline (offload_state_dict para no saturar RAM)...')
pipe = LTXPipeline.from_pretrained(
    'Lightricks/LTX-Video',
    torch_dtype=DTYPE,
    offload_state_dict=True,  # carga 1 componente a la vez, evita OOM en RAM de sistema
)

pipe.enable_model_cpu_offload()  # pesos en CPU, mueve a GPU solo durante forward
pipe.vae.enable_tiling()
print('Pipeline cargado OK')

# --- Config ---
prompt = 'A cinematic drone shot of a misty mountain range at sunrise, with golden light piercing through clouds.'
negative_prompt = 'worst quality, inconsistent motion, blurry, jittery, distorted'
width, height = 640, 480
num_frames = 33
num_steps = 30
guidance_scale = 5.0

print(f'Prompt: {prompt}')
print(f'Generando {num_frames} frames ({width}x{height}), {num_steps} steps...')

with torch.inference_mode():
    output = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        width=width,
        height=height,
        num_frames=num_frames,
        num_inference_steps=num_steps,
        guidance_scale=guidance_scale,
    )

video_frames = output.frames[0]
print(f'Frames generados: {len(video_frames)}')

t2v_path = '/content/ltx_t2v.mp4'
export_to_video(video_frames, t2v_path, fps=24)
print(f'Video guardado: {t2v_path}')

Video(t2v_path, embed=True, width=512)

del pipe
gc.collect()
torch.cuda.empty_cache()

## Image-to-Video con LTXConditionPipeline

In [ ]:
# ---- 4. IMAGE-TO-VIDEO ----

gc.collect()
torch.cuda.empty_cache()

print('Cargando LTXConditionPipeline (offload_state_dict)...')
pipe = LTXConditionPipeline.from_pretrained(
    'Lightricks/LTX-Video',
    torch_dtype=DTYPE,
    offload_state_dict=True,
)

pipe.enable_model_cpu_offload()
pipe.vae.enable_tiling()
print('Pipeline cargado OK')

# --- Cargar imagen de condicion ---
image_url = 'https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/guitar-man.png'
condition_image = load_image(image_url)
print(f'Imagen cargada: {condition_image.size}')
display(condition_image)

condition = LTXVideoCondition(image=condition_image, frame_index=0)

prompt = 'A man with short gray hair plays a red electric guitar, slow motion, cinematic lighting.'
negative_prompt = 'worst quality, inconsistent motion, blurry, jittery, distorted'

print(f'Prompt: {prompt}')
print('Generando...')

with torch.inference_mode():
    output = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        conditions=[condition],
        width=640,
        height=480,
        num_frames=33,
        num_inference_steps=30,
        guidance_scale=5.0,
    )

video_frames = output.frames[0]
print(f'Frames generados: {len(video_frames)}')

i2v_path = '/content/ltx_i2v.mp4'
export_to_video(video_frames, i2v_path, fps=24)
print(f'Video guardado: {i2v_path}')

Video(i2v_path, embed=True, width=512)

del pipe
gc.collect()
torch.cuda.empty_cache()

## Variante rápida: LTX 2.3 Distilled (8 steps)

In [ ]:
# ---- 5. (OPCIONAL) LTX 2.3 DISTILLED - 8 STEPS ----

# NOTA: Este es un modelo diferente (LTX-2.3) que requiere diffusers desde main.
# Solo ejecutar si quieres probar la version mas rapida.

FAST_MODE = input('Probar LTX-2.3 Distilled (8 steps)? (s/N): ').strip().lower() == 's'

if FAST_MODE:
    !pip install -qU git+https://github.com/huggingface/diffusers.git
    
    from diffusers import LTX2Pipeline

    print('Cargando LTX-2.3 Distilled...')
    pipe = LTX2Pipeline.from_pretrained(
        'diffusers/LTX-2.3-Distilled-Diffusers',
        torch_dtype=torch.bfloat16,
    )
    pipe.to('cuda')

    prompt = 'A flowing river in a forest at golden hour, gentle wind in the leaves.'
    print(f'Generando (8 steps)...')

    video, audio = pipe(
        prompt=prompt,
        width=768,
        height=512,
        num_frames=49,
        frame_rate=24,
        num_inference_steps=8,
        guidance_scale=1.0,
        output_type='np',
        return_dict=False,
    )

    from diffusers.pipelines.ltx2.export_utils import encode_video
    fast_path = '/content/ltx_fast.mp4'
    encode_video(video[0], fps=24, audio=audio[0].float().cpu(),
                 audio_sample_rate=pipe.vocoder.config.output_sampling_rate,
                 output_path=fast_path)
    print(f'Video guardado: {fast_path}')
    Video(fast_path, embed=True, width=512)
else:
    print('Omitido')

## Limpieza

In [ ]:
# ---- 6. LIMPIEZA ----
gc.collect()
torch.cuda.empty_cache()
print('Done')